# 02 — kNN: fuerza bruta vs. branch and bound
Compara tiempos y mide el efecto de la dimensionalidad sobre ULB.
Requiere haber corrido el 01. Corre todas las celdas.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")

def guardar_csv(resultados, ruta):
    if not resultados: return
    cols = list(resultados[0].keys())
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(",".join(cols) + "\n")
        for r in resultados:
            f.write(",".join(str(r[c]) for c in cols) + "\n")


In [ ]:
import numpy as np, os, time
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (recall_score,f1_score,roc_auc_score,average_precision_score,accuracy_score)
DATOS="datos_procesados/ulb.npz"; K=5; DIMS=[5,10,15,20,None]
carpeta=iniciar_ejecucion("02_knn_bb",{"datos":DATOS,"k":K})
d=np.load(DATOS,allow_pickle=True)
Xtr,ytr,Xte,yte=d["X_train"],d["y_train"],d["X_test"],d["y_test"]
log(f"Datos: train={Xtr.shape} test={Xte.shape} fraude_test={int(yte.sum())}",carpeta)

def evaluar(alg,nd=None):
    a,b=(Xtr,Xte) if nd is None else (Xtr[:,:nd],Xte[:,:nd])
    m=KNeighborsClassifier(n_neighbors=K,metric="euclidean",algorithm=alg,n_jobs=-1)
    t0=time.perf_counter(); m.fit(a,ytr); tf=time.perf_counter()-t0
    t0=time.perf_counter(); p=m.predict_proba(b)[:,1]; tp=time.perf_counter()-t0
    pr=(p>=0.5).astype(int)
    return {"algoritmo":alg,"dims":a.shape[1],"t_construccion_s":round(tf,3),
            "t_inferencia_s":round(tp,3),"ms_por_consulta":round(1000*tp/len(b),4),
            "accuracy":round(accuracy_score(yte,pr),4),"recall":round(recall_score(yte,pr,zero_division=0),4),
            "f1":round(f1_score(yte,pr,zero_division=0),4),"auc_roc":round(roc_auc_score(yte,p),4),
            "auc_pr":round(average_precision_score(yte,p),4)}

def tabla(res):
    cols=list(res[0].keys()); w={c:max(len(c),max(len(str(r[c])) for r in res)) for c in cols}
    log(" | ".join(c.ljust(w[c]) for c in cols),carpeta)
    log("-+-".join("-"*w[c] for c in cols),carpeta)
    for r in res: log(" | ".join(str(r[c]).ljust(w[c]) for c in cols),carpeta)

In [ ]:
log("\n--- Todas las caracteristicas ---",carpeta)
r1=[]
for alg in ["brute","kd_tree","ball_tree"]:
    log(f"Corriendo {alg}...",carpeta); r1.append(evaluar(alg))
tabla(r1); guardar_csv(r1,os.path.join(carpeta,"benchmark_completo.csv"))

log("\n--- Efecto de la dimensionalidad ---",carpeta)
r2=[]
for nd in DIMS:
    ndv=Xtr.shape[1] if nd is None else nd
    for alg in ["brute","kd_tree"]: r2.append(evaluar(alg,ndv))
tabla(r2); guardar_csv(r2,os.path.join(carpeta,"benchmark_dimensionalidad.csv"))

In [ ]:
plt.figure(figsize=(8,5))
for alg,mk in [("brute","o-"),("kd_tree","s-")]:
    pares=sorted((r["dims"],r["ms_por_consulta"]) for r in r2 if r["algoritmo"]==alg)
    plt.plot([p[0] for p in pares],[p[1] for p in pares],mk,label=alg)
plt.xlabel("Numero de caracteristicas"); plt.ylabel("ms por consulta")
plt.title("Fuerza bruta vs. branch and bound (kd-tree)"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(carpeta,"cruce_dimensionalidad.png"),dpi=150,bbox_inches="tight"); plt.show()
log(f"\nGuardado en: {carpeta}",carpeta)